# House Price — Analyse Exploratoire

Ce notebook réalise l'exploration complète du dataset *Ames Housing* (1 460 maisons, 81 variables) mis à disposition par Laplace Immo.

**Objectif** : comprendre la structure des données, identifier les features pertinentes et produire les visualisations nécessaires à la modélisation.

## 1. Librairies & Configuration

In [ ]:
# reload modules before executing user code.
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import dill
import matplotlib.pyplot as plt
import missingno as msno
import mlflow
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import pendulum
import ppscore as pps
import seaborn as sns
from loguru import logger
# from pycaret.regression import *
from sklearn import set_config
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (r2_score,
                             mean_squared_error,
                             mean_absolute_percentage_error,
                             max_error,
                            )
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from ydata_profiling import ProfileReport
from yellowbrick.regressor import ResidualsPlot


sys.path.append(str(Path.cwd().parent))
from settings.params import (DATA_DIR_INPUT,
                              DATA_DIR_OUTPUT,
                              MODEL_PARAMS,
                              REPORT_DIR,
                              TIMEZONE,
                              MODEL_NAME,
                              SEED
                             )
from src.make_dataset import load_data
from src.trainer import Trainer

set_config(display="diagram", print_changed_only=False)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# time in UTC
log_fmt = ("<green>{time:YYYY-MM-DD HH:mm:ss.SSS!UTC}</green> | <level>{level: <8}</level> | "
           "<cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - {message}"
          )
log_config = {
    "handlers": [
        {"sink": sys.stderr, "format": log_fmt},
    ],
}
logger.configure(**log_config)


In [ ]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = Path(PROJECT_DIR, 'data')
DATA_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR = Path(PROJECT_DIR, 'models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
DATA_DIR_INPUT = Path(DATA_DIR, 'input')
DATA_DIR_INPUT.mkdir(exist_ok=True, parents=True)
DATA_DIR_OUTPUT = Path(DATA_DIR, 'output')
DATA_DIR_OUTPUT.mkdir(exist_ok=True, parents=True)

logger.info(f"\nProject directory: {PROJECT_DIR} \nData dir: {DATA_DIR} \nModel dir: {MODEL_DIR}")

In [ ]:
TARGET_NAME = MODEL_PARAMS["TARGET"]
logger.info(f"Target name: {TARGET_NAME}")

## 2. Chargement des données

Les données proviennent du dataset **house_prices** disponible sur OpenML. La fonction `load_data` enrichit le dataset avec trois variables d'âge calculées à partir de la date de vente.

In [ ]:
data = load_data(dataset_name="house_prices", columns_to_lower=True)

In [ ]:
data.head()

In [ ]:
data = data.astype({
    'overallqual':str, 
    'overallcond':str,
    'garageyrblt':str,
    'yearbuilt':str,
    'yearremodadd':str,
    'mssubclass':str,
    'mosold':str,
    'yrsold':str
})

In [ ]:
data.info()

In [ ]:
data.describe(include="all")

## 3. Qualité des données — Valeurs manquantes

Analyse de la complétude de chaque variable. Les variables avec plus de 50% de valeurs manquantes seront exclues de la modélisation.

In [ ]:
msno.bar(data,
         filter="top",  # select only features that have a completion rate >= p
         p=MODEL_PARAMS["MIN_COMPLETION_RATE"] # filter columns with % of missing values > 50%
        );

## 4. Variable cible : `saleprice`

Analyse de la distribution du prix de vente. Une distribution asymétrique à droite (*right-skewed*) est typique des données immobilières.

In [ ]:
# Target: stat description
data[TARGET_NAME].describe()

In [ ]:
# Target distribution: raw vs log (box-cox transformation)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))

sns.histplot(data[TARGET_NAME], color='r', kde=True, ax=axes[0])
axes[0].set_title('Distribution of house prices')

sns.histplot(np.log(data[TARGET_NAME]), color='b', kde=True, ax=axes[1])
axes[1].set_title('Distribution of house prices in $log$ scale')
axes[1].set_xscale('log');

## 5. Variables catégorielles

### 5.1 Fréquences des modalités

Pour chaque variable qualitative, on observe la distribution des modalités.

In [ ]:
categorical_features = data.select_dtypes(include=["object", "bool"]).columns
logger.info(f"Categorical features:\n {categorical_features}\n")

numerical_features = data.select_dtypes(include="number").columns
logger.info(f"Numerical features:\n {numerical_features}")

In [ ]:
# Valeurs uniques et frequences pour chaque variable qualitative (incluant les nulls)
for col in categorical_features:
    freq = data[col].value_counts(dropna=False).rename_axis("Valeur").reset_index(name="Frequence")
    logger.info(f"\nVariable : {col} (nulls : {data[col].isna().sum()})\n{freq.to_string(index=False)}")

### 5.2 Analyse bivariée : variables catégorielles vs `saleprice`

Boxplots montrant l'impact de chaque variable catégorielle sur le prix de vente.

In [ ]:
import math

In [ ]:
# Analyse bivariee des variables categorielles avec la cible
target = "saleprice"
ncols = 1
nrows = math.ceil(len(categorical_features))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, nrows * 5))
axes = axes.ravel()

for i, col in enumerate(categorical_features):
    data.boxplot(column=target, by=col, ax=axes[i], grid=False)
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].set_ylabel(target)
    axes[i].tick_params(axis="x", rotation=45)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Boxplots : variables categorielles vs saleprice", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Variables numériques

### 6.1 Distributions univariées

In [ ]:
# Histogrammes des variables numeriques (hors cible et id)
num_cols = [col for col in numerical_features if col not in [target, "id"]]

ncols = 4
nrows = math.ceil(len(num_cols) / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, nrows * 4))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    axes[i].hist(data[col].dropna(), bins=30, edgecolor="black", color="steelblue", alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequence")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Distribution des variables numeriques", fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Analyse bivariée : variables numériques vs `saleprice`

Scatter plots pour identifier les relations linéaires entre les features numériques et le prix.

In [ ]:
# Scatter plots : chaque variable numerique vs saleprice
ncols = 4
nrows = math.ceil(len(num_cols) / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, nrows * 4))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    axes[i].scatter(data[col], data[target], alpha=0.3, s=10, color="steelblue")
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(target)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Scatter plots : variables numeriques vs saleprice", fontsize=14)
plt.tight_layout()
plt.show()

### 6.3 Matrice de corrélation

Identification des corrélations fortes entre variables numériques.

In [ ]:
# Matrice de correlation (variables numeriques)
import numpy as np

corr_matrix = data[list(numerical_features)].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_title("Matrice de correlation - variables numeriques", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations avec saleprice
correlations_with_target = (
    corr_matrix[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
)
logger.info(f"Correlations avec {target}:{correlations_with_target.to_string()}")

## 7. Rapport de profiling automatique

Génération d'un rapport HTML complet via `ydata-profiling` pour une exploration exhaustive du dataset.

In [ ]:
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)

profile = ProfileReport(data, title="House price - Report")
# profile.to_notebook_iframe()
profile.to_file(Path(REPORT_DIR, 'house_price_profiling.html'))

---
**Conclusions EDA** :
- La variable cible `saleprice` présente une distribution asymétrique à droite.
- Plusieurs variables ont un fort taux de valeurs manquantes (ex: `poolqc`, `alley`, `miscfeature` > 90%).
- Les variables les plus corrélées avec le prix sont `overallqual`, `grlivarea`, `garagecars`.
- Le feature engineering (âges : `building_age`, `remodel_age`, `garage_age`) enrichit le pouvoir prédictif.

➡️ La suite dans `house_price_02_modeling.ipynb`.